Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)
print('Library siap!')

Install Library Tambahan

In [ ]:
!pip install openpyxl imbalanced-learn

Load Si Gamis

In [ ]:
df_sigamis = pd.read_excel('Data/Si_Gamis_Rapih.xlsx', sheet_name=0)
print(f'Si Gamis: {df_sigamis.shape[0]:,} baris, {df_sigamis.shape[1]} kolom')
df_sigamis.head(3)

Load Padat Karya

In [ ]:
df_padatkarya = pd.read_excel('Data/Padat_Karya_Fix_Rapih.xlsx', sheet_name=0)
print(f'Padat Karya: {df_padatkarya.shape[0]:,} baris, {df_padatkarya.shape[1]} kolom')

kk_pk = df_padatkarya[df_padatkarya['SHDK'] == 'KEPALA KELUARGA'].copy().reset_index(drop=True)
print(f'\nTotal KK Padat Karya: {len(kk_pk):,}')
print('\nSTATUS AKHIR:')
print(kk_pk['STATUS AKHIR'].value_counts(dropna=False))
kk_pk[['ALAMAT','PEKERJAAN MANDIRI','PENDAPATAN PERKAPITA','STATUS AKHIR']].head(3)

Merging

In [ ]:
def clean_text(col):
    return (
        col.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.replace(r'[^A-Z0-9 ]', '', regex=True)
    )

# ── Si Gamis: ambil KK ────────────────────────────────────────────────────────
df_kk = df_sigamis[df_sigamis['HUBUNGAN'] == 'KEPALA KELUARGA'].copy().reset_index(drop=True)
df_kk['ALAMAT_KEY'] = clean_text(df_kk['ALAMAT'])

# Drop kolom PEKERJAAN dari Si Gamis agar tidak konflik saat merge
df_kk.drop(columns=['PEKERJAAN'], inplace=True, errors='ignore')

# Hitung JML_JIWA per KK
kk_indices = df_sigamis[df_sigamis['HUBUNGAN'] == 'KEPALA KELUARGA'].index.tolist()
jiwa_list = []
for i, idx in enumerate(kk_indices):
    next_idx = kk_indices[i+1] if i+1 < len(kk_indices) else len(df_sigamis)
    jiwa_list.append(next_idx - idx)
df_kk['JML_JIWA_HITUNG'] = jiwa_list

# ── Padat Karya: siapkan kolom untuk merge ────────────────────────────────────
kk_slim = kk_pk[['ALAMAT', 'NO RW', 'NO RT', 'PEKERJAAN MANDIRI',
                  'TOTAL PENDAPATAN', 'PENDAPATAN PERKAPITA', 'STATUS AKHIR',
                  'BPNT', 'PKH', 'APBD', 'KAMPUNG MADANI']].copy()

kk_slim.columns = ['ALAMAT_KEY', 'RW_PK', 'RT_PK',
                   'PEKERJAAN', 'TOTAL_PENDAPATAN', 'PENDAPATAN_PERKAPITA_ASLI', 'STATUS_AKHIR',
                   'BPNT', 'PKH', 'APBD', 'KAMPUNG_MADANI']
kk_slim['ALAMAT_KEY'] = clean_text(kk_slim['ALAMAT_KEY'])
kk_slim = kk_slim.drop_duplicates(subset='ALAMAT_KEY', keep='first')

# ── Merge ─────────────────────────────────────────────────────────────────────
df = df_kk.merge(kk_slim, on='ALAMAT_KEY', how='left')
df.drop(columns=['ALAMAT_KEY'], inplace=True)

# Gunakan JML_JIWA dari Si Gamis, fallback ke hitung manual
df['JML_JIWA'] = pd.to_numeric(df['JML_JIWA'], errors='coerce')
df['JML_JIWA'] = df['JML_JIWA'].fillna(df['JML_JIWA_HITUNG'])
df['RW'] = pd.to_numeric(df['RW_PK'], errors='coerce')
df['RT'] = pd.to_numeric(df['RT_PK'], errors='coerce')

# ── PENDAPATAN PERKAPITA ──────────────────────────────────────────────────────
df['TOTAL_PENDAPATAN'] = pd.to_numeric(df['TOTAL_PENDAPATAN'], errors='coerce').fillna(0)
df['PENDAPATAN_PERKAPITA'] = np.where(df['JML_JIWA'] > 0,
                                      df['TOTAL_PENDAPATAN'] / df['JML_JIWA'],
                                      df['TOTAL_PENDAPATAN'])

# ── HISTORY_BANTUAN dari BPNT, PKH, APBD, KAMPUNG_MADANI ─────────────────────
for col in ['BPNT', 'PKH', 'APBD', 'KAMPUNG_MADANI']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

kondisi_sudah = (df['BPNT'] > 0) | (df['PKH'] > 0) | \
                (df['APBD'] > 0) | (df['KAMPUNG_MADANI'] > 0)
df['HISTORY_BANTUAN'] = np.where(kondisi_sudah, 'Sudah', 'Belum')

# ── Kolom final ───────────────────────────────────────────────────────────────
df.drop(columns=['RW_PK', 'RT_PK'], inplace=True, errors='ignore')
df = df[['NO_KK', 'NIK', 'NAMA', 'ALAMAT', 'RW', 'RT',
         'JML_JIWA', 'IJAZAH_TERAKHIR',
         'PEKERJAAN', 'TOTAL_PENDAPATAN', 'PENDAPATAN_PERKAPITA',
         'BPNT', 'PKH', 'APBD', 'KAMPUNG_MADANI',
         'HISTORY_BANTUAN', 'STATUS_AKHIR']].copy()

print(f'Hasil merge           : {df.shape[0]:,} baris')
print(f'Match Padat Karya     : {df["STATUS_AKHIR"].notna().sum():,}')
print(f'Tidak match (missing) : {df["STATUS_AKHIR"].isna().sum():,}')
print('\nDistribusi STATUS_AKHIR:')
print(df['STATUS_AKHIR'].value_counts(dropna=False))

In [ ]:
# Tampilkan kolom yang relevan saja (df hasil merge punya banyak kolom)
kolom_tampil = ['ALAMAT', 'RW', 'RT', 'JML_JIWA', 'IJAZAH_TERAKHIR', 
                'PEKERJAAN', 'PENDAPATAN_PERKAPITA', 'HISTORY_BANTUAN', 'STATUS_AKHIR']

# Ambil 3 baris yang BERHASIL match (ada STATUS_AKHIR-nya), bukan yang NaN
df[df['STATUS_AKHIR'].notna()][kolom_tampil].head(3)

In [ ]:
# Cek berapa banyak alamat unik sebenarnya di antara 2.129 KK yang match
matched = df[df['STATUS_AKHIR'].notna()]
print(f"Total baris matched: {len(matched)}")
print(f"Jumlah alamat unik di antara yang matched: {matched['ALAMAT'].nunique()}")
print(f"\nContoh alamat yang dipakai berkali-kali:")
print(matched['ALAMAT'].value_counts().head(10))

In [ ]:
# Cek langsung di df_kk (Si Gamis level KK), SEBELUM merge
print(df_kk['ALAMAT'].value_counts().head(10))
print(f"\nJumlah baris df_kk dengan alamat 'KALIMAS BARU 1/60' (atau variasi mirip):")
print(df_kk[df_kk['ALAMAT'].str.contains('KALIMAS BARU', case=False, na=False)][['NO_KK','NAMA','ALAMAT']].head(20))

Statistik Deskriptif

In [ ]:
cek = df_kk[df_kk['ALAMAT'] == 'KALIMAS BARU 1/60']
print(f"Jumlah baris: {len(cek)}")
print(cek[['NO_KK','NIK','NAMA','ALAMAT']].to_string())

In [ ]:
print('=== Statistik Deskriptif Variabel Numerik ===')
print(df[['JML_JIWA','PENDAPATAN_PERKAPITA']].describe().round(2))

print('\n=== Distribusi Variabel Kategorikal ===')
print('\nIJAZAH_TERAKHIR:')
print(df['IJAZAH_TERAKHIR'].value_counts(dropna=False))
print('\nPEKERJAAN (Top 15):')
print(df['PEKERJAAN'].value_counts(dropna=False).head(15))
print('\nSTATUS_AKHIR (Label):')
print(df['STATUS_AKHIR'].value_counts(dropna=False))

Visualisasi Eksplorasi Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ijazah_order = ['Tidak punya ijazah','SD/sederajat','SMP/sederajat',
                'SMA/sederajat','D1/D2/D3','D4/S1','S2/S3']
ijazah_counts = df['IJAZAH_TERAKHIR'].value_counts().reindex(
    ijazah_order, fill_value=0)
axes[0].barh(ijazah_counts.index, ijazah_counts.values, color='#378ADD')
axes[0].set_title('Distribusi Ijazah Terakhir KK')
axes[0].set_xlabel('Jumlah KK')

top_pekerjaan = df['PEKERJAAN'].value_counts().head(10)
axes[1].barh(top_pekerjaan.index, top_pekerjaan.values, color='#534AB7')
axes[1].set_title('Top 10 Pekerjaan KK')
axes[1].set_xlabel('Jumlah KK')

status_counts = df['STATUS_AKHIR'].value_counts()
axes[2].bar(status_counts.index, status_counts.values,
            color=['#D85A30','#EF9F27','#1D9E75'])
axes[2].set_title('Distribusi Status Akhir (Label)')
axes[2].set_ylabel('Jumlah KK')

plt.tight_layout()
plt.savefig('eksplorasi_data.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: eksplorasi_data.png')

Cek Missing Value

In [ ]:
missing = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    '% Missing': (df.isnull().sum() / len(df) * 100).round(1)
})
print('=== Missing Value ===')
print(missing[missing['Jumlah Missing'] > 0])
print(f'\n=== Duplikat ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

Box Plot Outlier

In [ ]:
df_valid = df[df['PENDAPATAN_PERKAPITA'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df_valid['PENDAPATAN_PERKAPITA'].dropna(),
                patch_artist=True,
                boxprops=dict(facecolor='#BDD7EE'))
axes[0].set_title('Box Plot — Pendapatan Per Kapita')
axes[0].set_ylabel('Rp/bulan')
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))

axes[1].boxplot(df['JML_JIWA'].dropna(),
                patch_artist=True,
                boxprops=dict(facecolor='#BDD7EE'))
axes[1].set_title('Box Plot — Jumlah Jiwa per KK')
axes[1].set_ylabel('Jiwa')

plt.tight_layout()
plt.savefig('boxplot_outlier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: boxplot_outlier.png')

for col in ['PENDAPATAN_PERKAPITA', 'JML_JIWA']:
    vals = df[col].dropna()
    Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((vals < Q1 - 1.5*IQR) | (vals > Q3 + 1.5*IQR)).sum()
    print(f'{col}: Q1={Q1:,.0f}, Q3={Q3:,.0f}, IQR={IQR:,.0f} → Outlier: {outliers}')

Data Cleaning

In [ ]:
print(f'Baris sebelum cleaning: {len(df):,}')

# Drop baris tanpa STATUS_AKHIR (label wajib ada)
df_clean = df[df['STATUS_AKHIR'].notna()].copy()
print(f'Setelah drop tanpa STATUS_AKHIR : {len(df_clean):,}')

# Drop baris >50% kolom fitur kosong
fitur_cols = ['JML_JIWA','IJAZAH_TERAKHIR','PEKERJAAN','PENDAPATAN_PERKAPITA']
pct_missing = df_clean[fitur_cols].isnull().mean(axis=1)
df_clean = df_clean[pct_missing <= 0.5].copy()
print(f'Setelah drop >50% missing fitur : {len(df_clean):,}')

# Imputasi mean untuk numerik
mean_pendapatan = df_clean['PENDAPATAN_PERKAPITA'].mean()
mean_jiwa       = df_clean['JML_JIWA'].mean()
df_clean['PENDAPATAN_PERKAPITA'] = df_clean['PENDAPATAN_PERKAPITA'].fillna(mean_pendapatan)
df_clean['JML_JIWA']             = df_clean['JML_JIWA'].fillna(mean_jiwa)
print(f'\nImputasi mean PENDAPATAN_PERKAPITA : Rp {mean_pendapatan:,.0f}')
print(f'Imputasi mean JML_JIWA             : {mean_jiwa:.1f} jiwa')

# Imputasi modus untuk kategorikal
modus_ijazah    = df_clean['IJAZAH_TERAKHIR'].mode()[0]
modus_pekerjaan = df_clean['PEKERJAAN'].mode()[0]
df_clean['IJAZAH_TERAKHIR'] = df_clean['IJAZAH_TERAKHIR'].fillna(modus_ijazah)
df_clean['PEKERJAAN']       = df_clean['PEKERJAAN'].fillna(modus_pekerjaan)
print(f'Imputasi modus IJAZAH_TERAKHIR     : {modus_ijazah}')
print(f'Imputasi modus PEKERJAAN           : {modus_pekerjaan}')

# Hapus duplikat
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'\nTotal baris setelah cleaning : {len(df_clean):,}')
print(f'Missing value tersisa        : {df_clean[fitur_cols].isnull().sum().sum()}')

Encoding & Discretize Pendapatan

In [ ]:
# ── Encoding Ijazah sesuai Tabel 3.9 proposal ────────────────────────────────
ijazah_enc = {
    'Tidak punya ijazah': 0,
    'SD/sederajat'      : 1,
    'SMP/sederajat'     : 2,
    'SMA/sederajat'     : 3,
    'D1/D2/D3'          : 4,
    'D4/S1'             : 4,
    'S2/S3'             : 4
}
df_clean['IJAZAH_ENC'] = df_clean['IJAZAH_TERAKHIR'].map(ijazah_enc)
print('Mapping Ijazah:')
for k, v in ijazah_enc.items():
    print(f'  {v}: {k}')

# ── Encoding Pekerjaan ────────────────────────────────────────────────────────
le_pekerjaan = LabelEncoder()
df_clean['PEKERJAAN_ENC'] = le_pekerjaan.fit_transform(
    df_clean['PEKERJAAN'].astype(str))
print(f'\nJumlah kategori pekerjaan: {len(le_pekerjaan.classes_)}')

# ── Discretize Pendapatan Perkapita menjadi 4 kategori ───────────────────────
bins      = [0, 500000, 600000, 750000, float('inf')]
label_bin = [0, 1, 2, 3]
df_clean['PENDAPATAN_BIN'] = pd.cut(
    df_clean['PENDAPATAN_PERKAPITA'],
    bins=bins, labels=label_bin
)
# Pastikan tidak ada NaN setelah discretize
df_clean['PENDAPATAN_BIN'] = df_clean['PENDAPATAN_BIN'].fillna(0).astype(int)

print('\nMapping Pendapatan (Discretized):')
print('  0: < Rp 500.000       (Sangat Rendah)')
print('  1: Rp 500.000-600.000 (Rendah)')
print('  2: Rp 600.000-750.000 (Menengah)')
print('  3: > Rp 750.000       (Tinggi)')
print('\nDistribusi PENDAPATAN_BIN:')
print(df_clean['PENDAPATAN_BIN'].value_counts().sort_index().rename({
    0:'Sangat Rendah', 1:'Rendah', 2:'Menengah', 3:'Tinggi'}))

# ── Encoding Label 3 kelas ────────────────────────────────────────────────────
label_map = {'MISKIN': 0, 'PRA SEJAHTERA': 1, 'SEJAHTERA': 2}
df_clean['LABEL'] = df_clean['STATUS_AKHIR'].map(label_map)

unmapped = df_clean[df_clean['LABEL'].isna()]['STATUS_AKHIR'].unique()
if len(unmapped) > 0:
    print(f'\n⚠️ Nilai tidak terpetakan: {unmapped}')
    df_clean = df_clean[df_clean['LABEL'].notna()].copy()

df_clean['LABEL'] = df_clean['LABEL'].astype(int)

print(f'\nDistribusi Label:')
print(df_clean['LABEL'].value_counts().rename({
    0:'Miskin (0)', 1:'Pra Sejahtera (1)', 2:'Sejahtera (2)'}))

# Visualisasi distribusi label
labels_viz = ['Miskin', 'Pra Sejahtera', 'Sejahtera']
values_viz = [(df_clean['LABEL']==i).sum() for i in range(3)]
colors_viz = ['#D85A30','#EF9F27','#1D9E75']
plt.figure(figsize=(7, 4))
plt.bar(labels_viz, values_viz, color=colors_viz, width=0.5)
plt.title('Distribusi Label Kesejahteraan (3 Kelas)')
plt.ylabel('Jumlah KK')
for i, v in enumerate(values_viz):
    plt.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('distribusi_label.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: distribusi_label.png')

Pemilihan Fitur

In [ ]:
# 4 fitur sesuai proposal
features = ['JML_JIWA', 'IJAZAH_ENC', 'PEKERJAAN_ENC', 'PENDAPATAN_BIN']
target   = 'LABEL'

X = df_clean[features].fillna(0)
y = df_clean[target]

print('=== Dataset Siap Modeling ===')
print(f'Jumlah sampel  : {len(X):,}')
print(f'\nVariabel Independen (4 Fitur):')
print(f'  1. JML_JIWA       — tanggungan keluarga')
print(f'  2. IJAZAH_ENC     — tingkat pendidikan (0-4)')
print(f'  3. PEKERJAAN_ENC  — jenis pekerjaan (label encoded)')
print(f'  4. PENDAPATAN_BIN — pendapatan per kapita (0=sangat rendah, 3=tinggi)')
print(f'\nVariabel Dependen:')
print(f'  LABEL — 0=Miskin, 1=Pra Sejahtera, 2=Sejahtera')
print(f'\nDistribusi label:')
print(y.value_counts().rename({0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}))
X.describe().round(2)

Stratified Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Data Training : {len(X_train):,} sampel (80%)')
print(f'Data Testing  : {len(X_test):,} sampel (20%)')
print(f'\nDistribusi label Training:')
print(y_train.value_counts().rename({0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}))
print(f'\nDistribusi label Testing:')
print(y_test.value_counts().rename({0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}))
print(f'\nProporsi terjaga (stratified):')
for k, n in {0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}.items():
    ptr = (y_train==k).mean()*100
    pte = (y_test==k).mean()*100
    print(f'  {n}: Train={ptr:.1f}%, Test={pte:.1f}%')

 SMOTE (Oversampling)

In [ ]:
print('Distribusi SEBELUM SMOTE (training):')
print(pd.Series(y_train).value_counts().rename({
    0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}))

# Pastikan tidak ada NaN sebelum SMOTE
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

# k_neighbors=1 karena Sejahtera sangat sedikit di training
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nDistribusi SETELAH SMOTE (training):')
print(pd.Series(y_train_sm).value_counts().rename({
    0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'}))
print(f'\nTotal training setelah SMOTE : {len(X_train_sm):,}')
print('Test set TIDAK diubah — tetap data asli')

Build Model C4.5 

In [ ]:
# C4.5: criterion='entropy' (Information Gain)
model_c45 = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
model_c45.fit(X_train_sm, y_train_sm)

print('=== Model C4.5 ===')
print(f'Kedalaman pohon : {model_c45.get_depth()}')
print(f'Jumlah leaf     : {model_c45.get_n_leaves()}')
print()
print('Feature Importance:')
feat_names = ['Tanggungan Keluarga','Tingkat Pendidikan',
              'Pekerjaan','Pendapatan (Bin)']
for name, imp in zip(feat_names, model_c45.feature_importances_):
    print(f'  {name}: {imp:.4f}')

Visualisasi Decision Tree C4.5

In [ ]:
feature_names = ['Tanggungan Keluarga','Tingkat Pendidikan',
                 'Pekerjaan (Enc)','Pendapatan (Bin)']
class_names   = ['Miskin','Pra Sejahtera','Sejahtera']

plt.figure(figsize=(22, 10))
plot_tree(model_c45,
          feature_names=feature_names,
          class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree C4.5 — Klasifikasi Kesejahteraan Warga\nKelurahan Tanjung Perak',
          fontsize=14)
plt.savefig('decision_tree_c45.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: decision_tree_c45.png')

IF-THEN Rules C4.5

In [ ]:
feature_names = ['Tanggungan Keluarga','Tingkat Pendidikan',
                 'Pekerjaan (Enc)','Pendapatan (Bin)']
rules = export_text(model_c45, feature_names=feature_names)
print(rules)

 Pruning & Tuning C4.5 

In [ ]:
depths = range(2, 11)
acc_train_c45, acc_test_c45 = [], []

for d in depths:
    m = DecisionTreeClassifier(criterion='entropy', max_depth=d,
                               min_samples_split=10, min_samples_leaf=5, random_state=42)
    m.fit(X_train_sm, y_train_sm)
    acc_train_c45.append(accuracy_score(y_train_sm, m.predict(X_train_sm)))
    acc_test_c45.append(accuracy_score(y_test, m.predict(X_test)))

best_depth_c45 = list(depths)[acc_test_c45.index(max(acc_test_c45))]

print("max_depth | Akurasi Training | Akurasi Testing")
for d, tr, te in zip(depths, acc_train_c45, acc_test_c45):
    print(f"{d:9} | {tr*100:16.2f}% | {te*100:15.2f}%")

plt.figure(figsize=(9, 5))
plt.plot(depths, [a*100 for a in acc_train_c45], 'o-', label='Training', color='#378ADD')
plt.plot(depths, [a*100 for a in acc_test_c45], 'o-', label='Testing', color='#1D9E75')
plt.axvline(x=best_depth_c45, color='#D85A30', linestyle='--',
            label=f'Best depth = {best_depth_c45}')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy (%)')
plt.title('Pengaruh Max Depth terhadap Akurasi C4.5 (Pruning)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('pruning_tuning_c45.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best max_depth : {best_depth_c45}')
print(f'Akurasi testing : {max(acc_test_c45)*100:.2f}%')

 Model Final C4.5 & Evaluasi 

In [ ]:
model_c45_final = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=best_depth_c45,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
model_c45_final.fit(X_train_sm, y_train_sm)
y_pred_c45 = model_c45_final.predict(X_test)

print('=== Evaluasi Model Final C4.5 ===')
acc_c45  = accuracy_score(y_test, y_pred_c45)
prec_c45 = precision_score(y_test, y_pred_c45, average='weighted', zero_division=0)
rec_c45  = recall_score(y_test, y_pred_c45, average='weighted', zero_division=0)
f1_c45   = f1_score(y_test, y_pred_c45, average='weighted', zero_division=0)

print(f'Accuracy  : {acc_c45*100:.2f}%')
print(f'Precision : {prec_c45*100:.2f}%')
print(f'Recall    : {rec_c45*100:.2f}%')
print(f'F1-Score  : {f1_c45*100:.2f}%')
print()
print(classification_report(y_test, y_pred_c45,
      target_names=['Miskin','Pra Sejahtera','Sejahtera'],
      zero_division=0))

Max Depth 10

In [ ]:
# Gambar Decision Tree — pakai model FINAL (max_depth=10)
feature_names = ['Tanggungan Keluarga','Tingkat Pendidikan',
                 'Pekerjaan (Enc)','Pendapatan (Bin)']
class_names   = ['Miskin','Pra Sejahtera','Sejahtera']

plt.figure(figsize=(22, 10))
plot_tree(model_c45_final,
          feature_names=feature_names,
          class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree C4.5 — Klasifikasi Kesejahteraan Warga\nKelurahan Tanjung Perak',
          fontsize=14)
plt.savefig('decision_tree_c45_final.png', dpi=300, bbox_inches='tight')
plt.show()

IF-THEN Rules Max Depth 10

In [ ]:
# IF-THEN Rules — pakai model FINAL (max_depth=10)
rules_final = export_text(model_c45_final, feature_names=feature_names)
print(rules_final)

Confusion Matrix C4.5

In [ ]:
cm_c45 = confusion_matrix(y_test, y_pred_c45)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_c45, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Miskin','Pra Sejahtera','Sejahtera'],
            yticklabels=['Miskin','Pra Sejahtera','Sejahtera'])
plt.title('Confusion Matrix — C4.5')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.tight_layout()
plt.savefig('confusion_matrix_c45.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_c45.png')

 Build Model ID3 & Evaluasi

In [ ]:
# ID3: criterion='entropy', max_features=None, tidak ada pruning
# Perbedaan utama dari C4.5: ID3 tidak melakukan pruning dan
# tidak menangani atribut kontinu secara native
model_id3 = DecisionTreeClassifier(
    criterion='entropy',    # ID3 pakai Information Gain murni
    max_depth=None,         # ID3 tidak melakukan pruning
    min_samples_split=2,    # ID3 split seluas mungkin
    min_samples_leaf=1,
    random_state=42
)
model_id3.fit(X_train_sm, y_train_sm)
y_pred_id3 = model_id3.predict(X_test)

print('=== Evaluasi Model ID3 ===')
acc_id3  = accuracy_score(y_test, y_pred_id3)
prec_id3 = precision_score(y_test, y_pred_id3, average='weighted', zero_division=0)
rec_id3  = recall_score(y_test, y_pred_id3, average='weighted', zero_division=0)
f1_id3   = f1_score(y_test, y_pred_id3, average='weighted', zero_division=0)

print(f'Kedalaman pohon : {model_id3.get_depth()}')
print(f'Jumlah leaf     : {model_id3.get_n_leaves()}')
print(f'Accuracy  : {acc_id3*100:.2f}%')
print(f'Precision : {prec_id3*100:.2f}%')
print(f'Recall    : {rec_id3*100:.2f}%')
print(f'F1-Score  : {f1_id3*100:.2f}%')
print()
print(classification_report(y_test, y_pred_id3,
      target_names=['Miskin','Pra Sejahtera','Sejahtera'],
      zero_division=0))

Confusion Matrix ID3

In [ ]:
cm_id3 = confusion_matrix(y_test, y_pred_id3)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_id3, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Miskin','Pra Sejahtera','Sejahtera'],
            yticklabels=['Miskin','Pra Sejahtera','Sejahtera'])
plt.title('Confusion Matrix — ID3')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.tight_layout()
plt.savefig('confusion_matrix_id3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_id3.png')

 Build Model CART & Evaluasi

In [ ]:
# CART: criterion='gini' (Gini Impurity)
# Perbedaan dari C4.5: CART pakai Gini bukan Entropy/Gain Ratio
model_cart = DecisionTreeClassifier(
    criterion='gini',       # CART pakai Gini Impurity
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
model_cart.fit(X_train_sm, y_train_sm)
y_pred_cart = model_cart.predict(X_test)

print('=== Evaluasi Model CART ===')
acc_cart  = accuracy_score(y_test, y_pred_cart)
prec_cart = precision_score(y_test, y_pred_cart, average='weighted', zero_division=0)
rec_cart  = recall_score(y_test, y_pred_cart, average='weighted', zero_division=0)
f1_cart   = f1_score(y_test, y_pred_cart, average='weighted', zero_division=0)

print(f'Kedalaman pohon : {model_cart.get_depth()}')
print(f'Jumlah leaf     : {model_cart.get_n_leaves()}')
print(f'Accuracy  : {acc_cart*100:.2f}%')
print(f'Precision : {prec_cart*100:.2f}%')
print(f'Recall    : {rec_cart*100:.2f}%')
print(f'F1-Score  : {f1_cart*100:.2f}%')
print()
print(classification_report(y_test, y_pred_cart,
      target_names=['Miskin','Pra Sejahtera','Sejahtera'],
      zero_division=0))

Confusion Matrix CART

In [ ]:
cm_cart = confusion_matrix(y_test, y_pred_cart)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_cart, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Miskin','Pra Sejahtera','Sejahtera'],
            yticklabels=['Miskin','Pra Sejahtera','Sejahtera'])
plt.title('Confusion Matrix — CART')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.tight_layout()
plt.savefig('confusion_matrix_cart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_cart.png')

Tabel Perbandingan C4.5 vs ID3 vs CART

In [ ]:
hasil = pd.DataFrame({
    'Algoritma'    : ['C4.5', 'ID3', 'CART'],
    'Kriteria'     : ['Entropy (Gain Ratio)', 'Entropy (Info Gain)', 'Gini Impurity'],
    'Pruning'      : ['Ya (max_depth=5)', 'Tidak', 'Ya (max_depth=5)'],
    'Accuracy (%)'  : [acc_c45*100, acc_id3*100, acc_cart*100],
    'Precision (%)': [prec_c45*100, prec_id3*100, prec_cart*100],
    'Recall (%)'   : [rec_c45*100, rec_id3*100, rec_cart*100],
    'F1-Score (%)' : [f1_c45*100, f1_id3*100, f1_cart*100],
}).round(2)

print('=== Perbandingan Algoritma Decision Tree ===')
print(hasil.to_string(index=False))

# Visualisasi
x     = np.arange(3)
width = 0.2
metrics = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
colors  = ['#378ADD','#1D9E75','#D85A30','#534AB7']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, hasil[metric], width, label=metric, color=color)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(hasil['Algoritma'], fontsize=12)
ax.set_ylabel('Score (%)')
ax.set_title('Perbandingan Performa C4.5 vs ID3 vs CART')
ax.legend()
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('perbandingan_algoritma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: perbandingan_algoritma.png')

best_algo = hasil.loc[hasil['Accuracy (%)'].idxmax(), 'Algoritma']
print(f'\nAlgoritma terbaik berdasarkan Accuracy: {best_algo}')

 Anonimisasi

In [ ]:
def anonymize(val):
    if pd.isna(val):
        return None
    return 'ANON_' + hashlib.md5(str(val).encode()).hexdigest()[:8].upper()

df_clean['NO_KK_ANON']  = df_clean['NO_KK'].apply(anonymize)
df_clean['NIK_ANON']    = df_clean['NIK'].apply(anonymize)
df_clean['NAMA_ANON']   = ['Warga_' + str(i+1).zfill(4) for i in range(len(df_clean))]
df_clean['ALAMAT_ANON'] = df_clean['RW'].apply(
    lambda x: f'RW {int(x):02d}, Kel. Tanjung Perak'
    if pd.notna(x) else 'Tidak Diketahui')

print('Anonimisasi selesai!')
print(df_clean[['NO_KK_ANON','NIK_ANON','NAMA_ANON','ALAMAT_ANON']].head(5))

 Export Hasil Prediksi

In [ ]:
df_clean['PREDIKSI']       = model_c45_final.predict(X).tolist()
df_clean['PREDIKSI_LABEL'] = df_clean['PREDIKSI'].map({
    0:'Miskin', 1:'Pra Sejahtera', 2:'Sejahtera'})
df_clean['PROBABILITAS']   = model_c45_final.predict_proba(X).max(axis=1).round(3)

# Memasukkan TOTAL_PENDAPATAN dan HISTORY_BANTUAN ke export
df_export = df_clean[[
    'NO_KK_ANON','NIK_ANON','NAMA_ANON','ALAMAT_ANON',
    'RW','RT','JML_JIWA','IJAZAH_TERAKHIR','PEKERJAAN',
    'TOTAL_PENDAPATAN','PENDAPATAN_PERKAPITA','PENDAPATAN_BIN',
    'HISTORY_BANTUAN',
    'STATUS_AKHIR','LABEL','PREDIKSI_LABEL','PROBABILITAS'
]].copy()

df_export.rename(columns={
    'NO_KK_ANON' :'NO_KK',
    'NIK_ANON'   :'NIK',
    'NAMA_ANON'  :'NAMA',
    'ALAMAT_ANON':'ALAMAT'
}, inplace=True)

df_export.to_csv('hasil_prediksi_c45.csv', index=False)
print(f'File disimpan  : hasil_prediksi_c45.csv')
print(f'Total baris    : {len(df_export):,}')
print('\nDistribusi Prediksi:')
print(df_export['PREDIKSI_LABEL'].value_counts())
print('\n=== Ringkasan per RW ===')
rw_summary = df_export.groupby(['RW','PREDIKSI_LABEL']).size().unstack(fill_value=0)
rw_summary['TOTAL'] = rw_summary.sum(axis=1)
print(rw_summary)

In [ ]:
# 1. Filter warga yang diprediksi Miskin oleh model C4.5
calon_penerima = df_export[df_export['PREDIKSI_LABEL'] == 'Miskin'].copy()

# 2. Filter warga yang 'Belum' pernah menerima bantuan
calon_eligible = calon_penerima[calon_penerima['HISTORY_BANTUAN'] == 'Belum'].copy()

# 3. Urutkan berdasarkan tingkat kedaruratan:
#    - Pendapatan Perkapita terkecil (ascending=True)
#    - Jumlah Jiwa terbanyak (ascending=False)
prioritas_final = calon_eligible.sort_values(
    by=['PENDAPATAN_PERKAPITA', 'JML_JIWA'],
    ascending=[True, False]
)

print('\n=== 🌟 TOP 20 PRIORITAS PENERIMA BANTUAN ===')
# Tampilkan 20 urutan teratas
display(prioritas_final[['NAMA', 'RW', 'RT', 'PENDAPATAN_PERKAPITA', 'JML_JIWA', 'PREDIKSI_LABEL']].head(20))

# 4. Ekspor hasil prioritas ke file Excel khusus
prioritas_final.to_excel('Daftar_Prioritas_Bansos_Tanjung_Perak.xlsx', index=False)
print(f"\n✅ File rekomendasi disimpan : Daftar_Prioritas_Bansos_Tanjung_Perak.xlsx")
print(f"✅ Total kandidat layak menerima : {len(prioritas_final):,} KK")


print("\n\n--- Test Load Data CSV Dashboard ---")
df_test = pd.read_csv('hasil_prediksi_c45.csv')
print("Kolom tersedia:", df_test.columns.tolist())
print("Dimensi:", df_test.shape)
print(df_test.head(3))

In [ ]:
# cell 29
import joblib

# ── EXPORT MODEL UNTUK DASHBOARD STREAMLIT ──
# Menyimpan model klasifikasi final
joblib.dump(model_c45_final, 'model_c45.joblib')

# Menyimpan kamus pengubah teks pekerjaan ke angka (Label Encoder)
joblib.dump(le_pekerjaan, 'le_pekerjaan.joblib')

print("✅ Model C4.5 dan Label Encoder berhasil diexport!")
print("File 'model_c45.joblib' dan 'le_pekerjaan.joblib' siap digunakan oleh app.py")

In [ ]:
print(df.columns.tolist())